In [ ]:
import os
import datetime
import ollama

In [ ]:
def save_ai_story(prompt, filepath, model_name='llama3.1'):
    dir_name = os.path.dirname(filepath)
    if dir_name and not os.path.exists(dir_name):
        os.makedirs(dir_name)
        print(f"Created missing directory structure: {dir_name}")

    print(f"Querying Ollama ({model_name}) for: {os.path.basename(filepath)}...")
    
    # Pass the targeted model_name dynamically here
    response = ollama.generate(model=model_name, prompt=prompt)
    story_text = response['response']
    
    with open(filepath, 'w', encoding='utf-8') as file:
        file.write(story_text)
        
    print(f"Successfully generated and saved story to: {filepath}")
    return story_text

# Story Generator

Generates a 200-300 word story in English and Hebrew to be used for the quests and quizzes.

## Steps

1. English Narrative Generation via Llama
1. Hebrew Translation Optimization via DictaLM

In [ ]:
# Core story elements
level = "CEFR A1/A2"
character_name = "Faun"
animal_type = "faun"
gender = "Male"
quest_name = "garden_adventure"

# Sstory guidance variables
story_title = "יום הגינון של פאון (Faun's Gardening Day)"
story_overview = (
    "Faun has a strong רָעָב (hunger) for a sweet, crunchy treat. "
    "He decides to become a gardener for a day, taking a סַל (basket) "
    "of tools out to the sunny גַּן (garden). He carefully buries a small "
    "זֶרַע (seed) in the dirt, waters it, and proudly harvests his very "
    "first home-grown גֶּזֶר (carrot)."
)

# Python list of vocabulary words
word_list = [
    "רָעָב (hunger/hungry)",
    "סַל (basket)",
    "גַּן (garden)",
    "זֶרַע (seed)",
    "גֶּזֶר (carrot)"
]

# Format the word list for the prompt
formatted_words = "\n".join([f"- {word}" for word in word_list])

# English Generation Prompt
large_prompt_en = f"""
You are an expert children's book author. Write an original, charming short story in simple, clear English based on this plot overview: 
"{story_overview}"

Rules:
- Keep the sentences short and clear so they translate cleanly into early-intermediate language structures.
- Do not create any other names, use nouns only (bird, girl, city, etc).
- Do not explain grammar or add any extra text to the story, aside from the story text itself.
- Do not format any text in the story aside from using new lines.
"""

# Generate the unique timestamped filenames to prevent overriding
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
animal_folder = animal_type.lower().replace(" ", "_")
filename = f"{timestamp}_{quest_name}"

# Generate the English story using llama3.1
english_story_output = save_ai_story(
    prompt=large_prompt_en,
    filepath=f"./quests/{animal_folder}/{filename}_en.md",
    model_name='llama3.1'
)

# Construct the dynamic Hebrew translation prompt, feeding it the English text
large_prompt_he = f"""
You are an expert bilingual translator fluent in both English and Hebrew. 

Translate the following English story into grammatically correct, natural Hebrew suitable for a CEFR A1/A2 language learner. 

English Story to Translate:
\"\"\"
{english_story_output}
\"\"\"

Rules for the translation:
- Do NOT use vowel points (nikkud) at all. Write in clean, modern, unpointed Hebrew text (Ktav Male).
- Ensure strict gender agreement (since the main character {character_name} is {gender}, use proper masculine verb inflections and adjectives).
- Match the key concepts exactly to these Hebrew vocabulary items:
{formatted_words}
- Do not explain grammar, do not add conversational notes, and do not include the original English in your output. Return ONLY the Hebrew translation.

CRITICAL OUTPUT FORMATTING RULES:
- Provide ONLY the direct Hebrew translation.
- DO NOT include introductory remarks like "Here is the translation...".
- DO NOT include conversational text, pleasantries, or explanations.
- Start your response directly with the translated Hebrew title or the first line of the translated story text.
- Match the new lines from the English version.
"""

# Pass the dynamic translation prompt specifically to DictaLM, which has been built specifically for Hebrew
save_ai_story(
    prompt=large_prompt_he,
    filepath=f"./quests/{animal_folder}/{filename}_he.md",
    model_name='aminadaven/dictalm2.0-instruct:q4_k_m'
);